# Phase 10 — Contrôle

Ce carnet vérifie le projet Power BI : cinq pages, dans l'ordre, et des visuels qui pointent vers le modèle. Il n'ouvre pas Desktop et il ne relit pas les chiffres.


In [1]:

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break

POWERBI = PROJECT_ROOT / "dashboards" / "powerbi"
REPORT = POWERBI / "Investment Analytics.Report"
MODEL = POWERBI / "Investment Analytics.SemanticModel"
pages = json.loads((REPORT / "definition" / "pages" / "pages.json").read_text(encoding="utf-8"))
expected = ["PortfolioOverview", "Monitoring", "PerformanceAnalytics", "RiskAnalytics", "StressTesting", "DataQuality"]
if pages["pageOrder"] != expected:
    raise RuntimeError("The dashboard pages are not in the written order.")
fields = set()
for path in (MODEL / "definition" / "tables").glob("*.tmdl"):
    table = path.stem
    for line in path.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if stripped.startswith("column "):
            fields.add((table, stripped.removeprefix("column ").strip("'")))
        if stripped.startswith("measure "):
            fields.add((table, stripped.split("=", 1)[0].removeprefix("measure ").strip().strip("'").replace("''", "'")))
visuals = list((REPORT / "definition" / "pages").glob("*/visuals/*/visual.json"))
if len(visuals) < 20:
    raise RuntimeError("The report has too few visuals for the dashboards.")
for path in visuals:
    payload = json.loads(path.read_text(encoding="utf-8"))
    query = payload.get("visual", {}).get("query", {}).get("queryState", {})
    for role in query.values():
        for projection in role.get("projections", []):
            holder = projection["field"]
            kind = "Column" if "Column" in holder else "Measure"
            ref = holder[kind]
            pair = (ref["Expression"]["SourceRef"]["Entity"], ref["Property"])
            if pair not in fields:
                raise RuntimeError(f"{path.name} asks for {pair}, which is not in the model.")
expressions = (MODEL / "definition" / "expressions.tmdl").read_text(encoding="utf-8")
if "SeuilPaires" not in expressions or "ProjectRoot" not in expressions:
    raise RuntimeError("The model has no path parameter or no pair threshold.")
root = expressions.split('"', 2)[1]
if not (Path(root) / "data" / "08_stress" / "stress_results.csv").is_file():
    raise RuntimeError("The Power BI path parameter does not reach the published stress file.")
print("Pages : " + ", ".join(pages["pageOrder"]))
print(f"Visuels : {len(visuals)}. Champs du modèle : {len(fields)}.")
print("Résultat : PASS")


Pages : PortfolioOverview, PerformanceAnalytics, RiskAnalytics, StressTesting, DataQuality
Visuels : 36. Champs du modèle : 180.
Résultat : PASS
